# BRM on the house dataset (regression)

King County, WA house-sales regression. The Python package ships the full ~21,600-row dataset; for quick iteration we can ask `load_house` for a sample.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

from blockwise import BRM, simulate_blockwise_missing, datasets

In [ ]:
house = datasets.load_house()
print(house.shape)
house.head()

## Induce missingness, split, and fit

In [ ]:
house_miss = simulate_blockwise_missing(
    house,
    blocks=[
        ["sqft_living", "sqft_lot", "sqft_above"],
        ["bedrooms", "bathrooms", "floors", "grade"],
    ],
    prop_missing=0.30,
    noise=0.05,
)

X = house_miss.drop(columns=["price"])
y = house_miss["price"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1234)
print(X_train.shape, X_test.shape)

In [ ]:
brm_lm = BRM(estimator=LinearRegression()).fit(X_train, y_train)
pred_lm = brm_lm.predict(X_test)
rmse_lm = float(np.sqrt(np.mean((y_test - pred_lm) ** 2)))
print(f"n_blocks: {brm_lm.n_blocks_}")
print(f"BRM (lm) RMSE: {rmse_lm:,.0f}")

In [ ]:
brm_gb = BRM(
    estimator=GradientBoostingRegressor(random_state=0, n_estimators=200),
    n_blocks=brm_lm.n_blocks_,
).fit(X_train, y_train)
pred_gb = brm_gb.predict(X_test)
rmse_gb = float(np.sqrt(np.mean((y_test - pred_gb) ** 2)))
print(f"BRM (gbr) RMSE: {rmse_gb:,.0f}")

## Citation

Srinivasan, K., Currim, F., and Ram, S. (2025). *A Reduced Modeling Approach for Making Predictions With Incomplete Data Having Blockwise Missing Patterns.* INFORMS Journal on Data Science.